# RTK Wave Buoy — First Deployment
Parses `dataLog00072.ubx`, trims to the ocean-only segment, and plots position time series + map.

In [ ]:
# Install dependencies if missing
import subprocess, sys
for pkg in ['pyubx2', 'contextily', 'pyproj']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('Dependencies OK')

In [ ]:
import sys
sys.path.insert(0, '../ubx_parsers')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from datetime import datetime, timezone
from v3_ubx_parser import parse_ubx_file, analyze

UBX_FILE = 'dataLog00072.ubx'

RTK_NAMES  = {0: 'None', 1: 'Float', 2: 'Fix'}
RTK_COLORS = {0: '#aaaaaa', 1: '#f0a500', 2: '#2ca02c'}

## Parse

In [ ]:
positions = parse_ubx_file(UBX_FILE)
analyze(positions)

## Build full DataFrame

In [ ]:
df_all = pd.DataFrame(positions)

def row_to_dt(r):
    try:
        return datetime(int(r.year), int(r.month), int(r.day),
                        int(r.hour), int(r.minute), int(r.second),
                        tzinfo=timezone.utc)
    except Exception:
        return pd.NaT

df_all['datetime'] = df_all.apply(row_to_dt, axis=1)
df_all = df_all.dropna(subset=['datetime'])
df_all = df_all.sort_values('datetime').reset_index(drop=True)
df_all['rtk_label'] = df_all['carrier_solution'].map(RTK_NAMES)
df_all['rtk_color'] = df_all['carrier_solution'].map(RTK_COLORS)

print(f"Full dataset: {len(df_all)} records")
print(f"  {df_all['datetime'].min()} → {df_all['datetime'].max()}")

## Trim to ocean segment
The buoy was in the ocean when `longitude < -117.260°` and `altitude_msl < 10 m`.  
The lab (Scripps) sits at ~62 m elevation at lon ≈ -117.251°.

In [ ]:
# identify contiguous ocean block: lon west of -117.260 AND altitude < 10 m
ocean_mask = (df_all['longitude'] < -117.260) & (df_all['altitude_msl'] < 10)

ocean_start = df_all.loc[ocean_mask, 'datetime'].min()
ocean_end   = df_all.loc[ocean_mask, 'datetime'].max()

print(f"Ocean segment: {ocean_start}  →  {ocean_end}")
print(f"Duration: {ocean_end - ocean_start}")

df = df_all[ocean_mask].copy().reset_index(drop=True)
print(f"Ocean records: {len(df)}")
print(f"RTK breakdown:\n{df['rtk_label'].value_counts()}")

## Map — buoy track over satellite basemap

In [ ]:
import contextily as cx
import matplotlib.patches as mpatches
from pyproj import Transformer

# project lat/lon → Web Mercator (EPSG:3857) for contextily
transformer = Transformer.from_crs('EPSG:4326', 'EPSG:3857', always_xy=True)
x, y = transformer.transform(df['longitude'].values, df['latitude'].values)

fig, ax = plt.subplots(figsize=(10, 10))

for cs, label, color in [(0,'None','#aaaaaa'), (1,'Float','#f0a500'), (2,'Fix','#2ca02c')]:
    mask = df['carrier_solution'] == cs
    if mask.any():
        ax.scatter(x[mask], y[mask], s=6, c=color, label=f'RTK {label}',
                   alpha=0.8, linewidths=0)

# start / end markers
ax.plot(x[0],  y[0],  'b^', ms=10, zorder=5, label='Deploy start')
ax.plot(x[-1], y[-1], 'rs', ms=10, zorder=5, label='Deploy end')

cx.add_basemap(ax, crs='EPSG:3857', source=cx.providers.Esri.WorldImagery, zoom=16)

ax.set_xlabel('Easting (m, Web Mercator)')
ax.set_ylabel('Northing (m, Web Mercator)')
ax.set_title('RTK Wave Buoy Track — Ocean Deployment')
ax.legend(loc='upper right', markerscale=2)
plt.tight_layout()
plt.show()

## RTK Mode Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))

ax.step(df['datetime'], df['carrier_solution'], where='post',
        color='steelblue', linewidth=1.2)
ax.fill_between(df['datetime'], df['carrier_solution'],
                step='post', alpha=0.2, color='steelblue')

ax.set_yticks([0, 1, 2])
ax.set_yticklabels(['None', 'Float', 'Fix'])
ax.set_ylabel('RTK Mode')
ax.set_xlabel('UTC Time')
ax.set_title('RTK Carrier Solution — Ocean Deployment')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %H:%M'))
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## Latitude Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

sc = ax.scatter(df['datetime'], df['latitude'],
                c=df['carrier_solution'], cmap='RdYlGn',
                vmin=0, vmax=2, s=4, alpha=0.8)
cbar = plt.colorbar(sc, ax=ax, ticks=[0, 1, 2])
cbar.ax.set_yticklabels(['None', 'Float', 'Fix'])
cbar.set_label('RTK Mode')

ax.set_ylabel('Latitude (°)')
ax.set_xlabel('UTC Time')
ax.set_title('Latitude Over Time — Ocean Deployment')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %H:%M'))
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## Longitude Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

sc = ax.scatter(df['datetime'], df['longitude'],
                c=df['carrier_solution'], cmap='RdYlGn',
                vmin=0, vmax=2, s=4, alpha=0.8)
cbar = plt.colorbar(sc, ax=ax, ticks=[0, 1, 2])
cbar.ax.set_yticklabels(['None', 'Float', 'Fix'])
cbar.set_label('RTK Mode')

ax.set_ylabel('Longitude (°)')
ax.set_xlabel('UTC Time')
ax.set_title('Longitude Over Time — Ocean Deployment')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %H:%M'))
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## Altitude MSL Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

sc = ax.scatter(df['datetime'], df['altitude_msl'],
                c=df['carrier_solution'], cmap='RdYlGn',
                vmin=0, vmax=2, s=4, alpha=0.8)
cbar = plt.colorbar(sc, ax=ax, ticks=[0, 1, 2])
cbar.ax.set_yticklabels(['None', 'Float', 'Fix'])
cbar.set_label('RTK Mode')

ax.set_ylabel('Altitude MSL (m)')
ax.set_xlabel('UTC Time')
ax.set_title('Altitude MSL Over Time — Ocean Deployment')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %H:%M'))
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## Identify moored period
Tighten the mask to lon < -117.264° to exclude the transit legs (lab ↔ mooring).  
This isolates only the stationary moored segment.

In [ ]:
# Moored = west of -117.264° (excludes transit) AND altitude < 5 m
moored_mask = (df_all['longitude'] < -117.264) & (df_all['altitude_msl'] < 5)

moored_start = df_all.loc[moored_mask, 'datetime'].min()
moored_end   = df_all.loc[moored_mask, 'datetime'].max()
moored_start_itow = int(df_all.loc[moored_mask, 'timestamp'].min() * 1000)
moored_end_itow   = int(df_all.loc[moored_mask, 'timestamp'].max() * 1000)

df_moored = df_all[moored_mask].copy().reset_index(drop=True)

print(f"Moored start : {moored_start}  (iTOW {moored_start_itow})")
print(f"Moored end   : {moored_end}  (iTOW {moored_end_itow})")
print(f"Duration     : {moored_end - moored_start}")
print(f"Records      : {len(df_moored)}")
print(f"\nRTK breakdown:\n{df_moored['rtk_label'].value_counts()}")
print(f"\nPosition cluster:")
print(f"  Lat  {df_moored['latitude'].mean():.7f} ± {df_moored['latitude'].std()*111000*100:.1f} cm")
print(f"  Lon  {df_moored['longitude'].mean():.7f} ± {df_moored['longitude'].std()*111000*100:.1f} cm")
print(f"  Alt  {df_moored['altitude_msl'].mean():.3f} ± {df_moored['altitude_msl'].std()*100:.1f} cm")

## Write truncated UBX — moored period only
Reads the original UBX and writes only messages whose `iTOW` falls within the moored window.  
Messages without an iTOW field (e.g. RXM-RAWX, RXM-SFRBX) are included when they fall between two in-window messages.

In [ ]:
from pyubx2 import UBXReader

OUT_FILE = 'dataLog00072_moored.ubx'

in_window = False   # tracks whether current iTOW is inside moored window
pending   = []      # buffer for messages without iTOW (held until next iTOW-bearing message)
written   = 0
skipped   = 0

with open(UBX_FILE, 'rb') as fin, open(OUT_FILE, 'wb') as fout:
    ubr = UBXReader(fin, errorhandler=lambda e: None)

    for raw, parsed in ubr:
        if parsed is None:
            continue

        itow = getattr(parsed, 'iTOW', None)

        if itow is not None:
            in_window = moored_start_itow <= itow <= moored_end_itow
            if in_window:
                # flush any buffered no-iTOW messages that belong here
                for p in pending:
                    fout.write(p)
                    written += 1
                pending.clear()
                fout.write(raw)
                written += 1
            else:
                pending.clear()   # discard buffered msgs that preceded an out-of-window iTOW
                skipped += 1
        else:
            # no iTOW — buffer until we know the next iTOW
            pending.append(raw)
            if not in_window:
                skipped += len(pending)
                pending.clear()

import os
size_mb = os.path.getsize(OUT_FILE) / 1e6
print(f"Written : {written} messages  →  {OUT_FILE}  ({size_mb:.1f} MB)")
print(f"Skipped : {skipped} messages")